# `tckdb-client` builder + two-phase artifact demo

Builds a full CH3 + H → CH4 computed-reaction upload, attaches a
fake artifact to one TS calculation and one species-side
calculation, then walks the two-phase upload flow.

**Without `TCKDB_BASE_URL` / `TCKDB_API_KEY`** the notebook
exercises the builder layer locally and shows a *mock-IDs* plan
preview — no network access needed.

**With both env vars set** the final cell also performs the live
scientific upload and the second-phase artifact uploads.

A sibling Python script lives at
[`builder_computed_reaction_demo.py`](builder_computed_reaction_demo.py);
the two stay in lockstep and are covered by the
`tests/test_builder_computed_reaction_demo.py` smoke test.


## 1 · Imports and shared constants


In [ ]:
from __future__ import annotations

import json
import os
import tempfile
import time
import warnings
from pathlib import Path

from tckdb_client import TCKDBClient
from tckdb_client.builders import (
    Calculation,
    ChemReaction,
    ComputedReactionUpload,
    Geometry,
    Kinetics,
    LevelOfTheory,
    PlannedArtifactUpload,
    Species,
    SourceCalculations,
    SoftwareRelease,
    Statmech,
    Thermo,
    Transport,
    TransitionState,
)

SR = SoftwareRelease(software="Gaussian", version="16", revision="C.01")
LOT = LevelOfTheory(method="wb97xd", basis="def2tzvp")

CH3_XYZ = "4\nch3\nC 0 0 0\nH 0 0 1\nH 0 1 0\nH 0 -1 0"
H_XYZ   = "1\nh\nH 0 0 0"
CH4_XYZ = "5\nch4\nC 0 0 0\nH 0 0 1\nH 0 0 -1\nH 0 1 0\nH 0 -1 0"
TS_XYZ  = "3\nts\nC 0 0 0\nH 0 0 0.8\nH 0 0 -1.0"


## 2 · Materialise tiny stand-in artifact files

`Calculation.add_artifact` accepts any path — the file is read
lazily by `client.upload_artifacts(...)` during the second
phase. The notebook writes two small `.log` files into a
temporary directory so the same upload object can run offline
*and* fire real second-phase uploads when env vars are set.


In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="tckdb-builder-demo-"))
(workdir / "ts_opt.log").write_text(
    "Entering Gaussian System, Link 0\n"
    "fake TS opt output log — demo only\n"
)
(workdir / "ch4_sp.log").write_text(
    "Entering Gaussian System, Link 0\n"
    "fake CH4 sp output log — demo only\n"
)
artifact_paths = {
    "ts_opt": workdir / "ts_opt.log",
    "ch4_sp": workdir / "ch4_sp.log",
}
print("temp dir:", workdir)
for k, p in artifact_paths.items():
    print(f"  {k:>10}: {p}  ({p.stat().st_size} bytes)")

## 3 · Build the upload

Three species (CH3, H, CH4) each with opt + freq + sp; one TS
with opt + freq + sp; one modified-Arrhenius kinetics record
with duplicate `reactant_energy` source roles; per-species
thermo (NASA), statmech, and transport. Two artifacts are
attached — one TS-side, one CH4-side — to exercise both
artifact-plan paths.


In [ ]:
def calc_trio(label_prefix, xyz, sp_energy):
    geom = Geometry.from_xyz(xyz)
    opt = Calculation.opt(
        SR, LOT, output_geometry=geom, converged=True,
        final_energy_hartree=sp_energy - 0.05,
        label=f"{label_prefix} opt",
    )
    freq = Calculation.freq(
        SR, LOT, n_imag=0, zpe_hartree=0.03,
        depends_on=opt, label=f"{label_prefix} freq",
    )
    sp = Calculation.sp(
        SR, LOT, electronic_energy_hartree=sp_energy,
        depends_on=opt, label=f"{label_prefix} sp",
    )
    return opt, freq, sp


ch3_opt, ch3_freq, ch3_sp = calc_trio("ch3", CH3_XYZ, -39.71)
h_opt,   h_freq,   h_sp   = calc_trio("h",   H_XYZ,   -0.5)
ch4_opt, ch4_freq, ch4_sp = calc_trio("ch4", CH4_XYZ, -40.51)

ts_geom = Geometry.from_xyz(TS_XYZ)
ts_opt = Calculation.opt(
    SR, LOT, output_geometry=ts_geom, converged=True,
    final_energy_hartree=-40.45, label="ts opt",
)
ts_freq = Calculation.freq(
    SR, LOT, n_imag=1, imag_freq_cm1=-1200.0, zpe_hartree=0.04,
    depends_on=ts_opt, label="ts freq",
)
ts_sp = Calculation.sp(
    SR, LOT, electronic_energy_hartree=-40.42,
    depends_on=ts_opt, label="ts sp",
)

# Attach artifacts — local metadata only; bytes ship in phase 2.
ts_opt.add_artifact(artifact_paths["ts_opt"], kind="output_log")
ch4_sp.add_artifact(artifact_paths["ch4_sp"], kind="output_log")

ch3 = Species(smiles="[CH3]", charge=0, multiplicity=2, label="CH3")
h   = Species(smiles="[H]",   charge=0, multiplicity=2, label="H")
ch4 = Species(smiles="C",     charge=0, multiplicity=1, label="CH4")

# Kinetics-side sources: duplicate reactant_energy roles drop out of
# the list-of-tuples form naturally via the list kwarg.
kin_sources = SourceCalculations(
    reactant_energy=[ch3_sp, h_sp],
    product_energy=ch4_sp,
    ts_energy=ts_sp,
    freq=ts_freq,
)
kin = Kinetics.modified_arrhenius(
    A=1.2e13, A_units="cm3/mol/s", n=0.5, Ea=10.0, Ea_units="kJ/mol",
    Tmin=300, Tmax=2500,
    source_calculations=kin_sources.as_list(),
)

# CH4-side thermo + statmech share an opt/freq bag.
ch4_sources = SourceCalculations(opt=ch4_opt, freq=ch4_freq)

rxn = ChemReaction(
    reactants=[ch3, h], products=[ch4],
    family="H_Abstraction",
    transition_state=TransitionState(
        charge=0, multiplicity=2, geometry=ts_geom, label="ts",
    ),
    kinetics=[kin],
)

upload = ComputedReactionUpload(
    reaction=rxn,
    calculations=[ts_opt, ts_freq, ts_sp],
    species_calculations={
        ch3: [ch3_opt, ch3_freq, ch3_sp],
        h:   [h_opt,   h_freq,   h_sp],
        ch4: [ch4_opt, ch4_freq, ch4_sp],
    },
    species_thermo={
        ch4: Thermo.nasa(
            coeffs_low=[0.5] + [0.0] * 6,
            coeffs_high=[0.5] + [0.0] * 6,
            t_low=200, t_mid=1000, t_high=5000,
            h298_kj_mol=-74.6, s298_j_mol_k=186.3,
            source_calculations=ch4_sources.only("opt", "freq"),
        ),
    },
    species_statmech={
        ch4: Statmech(
            external_symmetry=12, point_group="Td",
            is_linear=False, rigid_rotor_kind="spherical_top",
            statmech_treatment="rrho",
            source_calculations=ch4_sources.only("opt", "freq"),
        ),
    },
    species_transport={
        ch4: Transport(
            sigma_angstrom=3.8, epsilon_over_k_k=141.4,
            dipole_debye=0.0, polarizability_angstrom3=2.6,
            rotational_relaxation=13.0,
        ),
    },
)
payload = upload.to_payload()
print("payload built:", len(payload["species"]), "species,",
      len(payload.get("kinetics", [])), "kinetics record(s)")

## 4 · Payload summary

What the bundle endpoint will receive. Note that **no artifact
bytes appear in the payload** — they're held on the builder
objects until phase 2.


In [ ]:
# Migrated from the demo-local ``payload_summary(payload)`` helper to
# ``upload.summary().to_text()`` — a small builder-side preview surface
# that does not depend on the payload dict shape. See
# ``docs/builder_summary_design.md`` for the design.
print(upload.summary().to_text())

## 5 · Emission diagnostics

Every warning here flags data the builder accepted locally but
that today's backend bundle schemas don't carry on the wire.
The two `artifact_upload_requires_second_phase` entries are the
cue to run the phase-2 cell at the bottom of the notebook.


In [ ]:
diags = upload.emission_diagnostics()
if not diags:
    print("(none)")
for diag in diags:
    print(f"[{diag.level.upper():>7}] {diag.code}")
    print(f"          path: {diag.path}")
    msg = diag.message.strip()
    for chunk in (msg[i:i + 70] for i in range(0, len(msg), 70)):
        print(f"          {chunk}")

## 6 · Artifact summary

What's attached to which calculation, in which bucket (TS or
per-species). These files don't move during the scientific
upload — only the metadata travels.


In [ ]:
entries = list(upload.iter_calculation_entries(with_artifacts_only=True))
if not entries:
    print("(no artifacts attached)")
else:
    total = sum(len(e.calculation.artifacts) for e in entries)
    print(f"{total} artifact(s) across {len(entries)} calculation(s):")
    for entry in entries:
        calc = entry.calculation
        for art in calc.artifacts:
            label = calc.label or calc.type
            print(f"  - [{entry.bucket:>4}] {label:<10} {art.kind:<18} {art.path}")
    print("\nArtifacts ride on a second POST per calc once the server returns "
          "calculation IDs — see the cell below.")


## 7 · Artifact plan preview (offline, mock IDs)

`upload.artifact_plan(result)` needs a real server response.
When the notebook runs without env vars, feed it a synthesised
`{calculation_key: id}` map so the plan shape is visible
anyway. This is exactly the shape `client.upload_artifacts` will
consume — `PlannedArtifactUpload(calculation_key, calculation_id,
path, kind, label, sha256, bytes)`.


In [ ]:
plan_preview = upload.artifact_plan_preview()
for entry in plan_preview:
    print(
        f"  - calc_key={entry.calculation_key:<10} "
        f"calc_id={entry.calculation_id:<6} "
        f"kind={entry.kind:<12} path={entry.path}"
    )


## 8 · Live upload (optional)

The cell below performs the real two-phase upload when both
`TCKDB_BASE_URL` and `TCKDB_API_KEY` are set. With either
missing the cell short-circuits and prints a friendly message —
no network call.


In [ ]:
base_url = os.environ.get("TCKDB_BASE_URL")
api_key  = os.environ.get("TCKDB_API_KEY")

if not base_url or not api_key:
    print("TCKDB_BASE_URL or TCKDB_API_KEY not set — skipping live upload.")
else:
    with TCKDBClient(base_url, api_key=api_key) as client:
        with warnings.catch_warnings():
            warnings.simplefilter("always")
            result = client.upload(upload, warn_on_dropped_fields=True)
        print("== Server response (truncated) ==")
        print(json.dumps(result, indent=2)[:1200])
        print()

        try:
            plan = upload.artifact_plan(result)
        except Exception as exc:
            print(f"artifact_plan failed: {exc}")
        else:
            idem_prefix = f"builder-demo:{int(time.time())}"
            print(f"== Artifact upload (phase 2, prefix={idem_prefix!r}) ==")
            if not plan:
                print("(no artifacts to upload)")
            else:
                artifact_results = client.upload_artifacts(
                    plan, idempotency_key_prefix=idem_prefix,
                )
                for entry, response in zip(plan, artifact_results):
                    n = len(response.get("artifacts", [])) if isinstance(response, dict) else 0
                    print(
                        f"  - calc_id={entry.calculation_id:<6} "
                        f"kind={entry.kind:<12} -> {n} server row(s)"
                    )

---

**Where to go next**

- The matching script `builder_computed_reaction_demo.py` is the
  same flow without a Jupyter kernel; it's what
  `tests/test_builder_computed_reaction_demo.py` runs in CI.
- The full API surface for the builder layer is documented at
  [`docs/builder_api_mvp.md`](../docs/builder_api_mvp.md).
- The public-beta stability contract lives at
  [`docs/builder_api_stability.md`](../docs/builder_api_stability.md).
